In [2]:
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import xgboost as xgb

# 1. Load Data
df = pd.read_csv('merged_nasa_pvgis_hourly.csv')
df['timestamp'] = pd.to_datetime(df['timestamp'])

# 2. Feature Engineering
df['hour'] = df['timestamp'].dt.hour
df['month'] = df['timestamp'].dt.month
df['dayofyear'] = df['timestamp'].dt.dayofyear

features = [
    'NASA_temperature_C',
    'NASA_relative_humidity_pct',
    'NASA_wind_speed_m_s',
    'NASA_precipitation_mm_per_hour',
    'PVGIS_direct_irradiance_W_m2',
    'PVGIS_diffuse_irradiance_W_m2',
    'PVGIS_sun_height_deg',
    'hour',
    'month',
    'dayofyear',
]
target = 'PVGIS_PV_power_W_for_1kWp_system'

X = df[features]
y = df[target]

# 3. Chronological Split (80% Train, 20% Test)
train_size = int(len(df) * 0.8)
X_train, X_test = X.iloc[:train_size], X.iloc[train_size:]
y_train, y_test = y.iloc[:train_size], y.iloc[train_size:]

# 4. Train XGBoost Model
xgb_model = xgb.XGBRegressor(
    n_estimators=150,
    learning_rate=0.08,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
)
xgb_model.fit(X_train, y_train)

# 5. Predictions
y_train_pred = xgb_model.predict(X_train)
y_test_pred = xgb_model.predict(X_test)

# 6. Evaluation Metrics (Error + Accuracy)
train_rmse = np.sqrt(mean_squared_error(y_train, y_train_pred))
test_rmse = np.sqrt(mean_squared_error(y_test, y_test_pred))
test_mae = mean_absolute_error(y_test, y_test_pred)

train_r2 = r2_score(y_train, y_train_pred)
test_r2 = r2_score(y_test, y_test_pred)

# Calculate Percentage Accuracy using R2 Score
train_accuracy = train_r2 * 100
test_accuracy = test_r2 * 100

print('--- Model Training & Testing Performance ---')
print(f'Train RMSE: {train_rmse:.2f} W')
print(f'Test RMSE:  {test_rmse:.2f} W')
print(f'Test MAE:   {test_mae:.2f} W')
print('--------------------------------------------')
print(f'Training Accuracy ($R^2$): {train_accuracy:.2f}%')
print(f'Testing Accuracy ($R^2$):  {test_accuracy:.2f}%')

--- Model Training & Testing Performance ---
Train RMSE: 2.94 W
Test RMSE:  4.94 W
Test MAE:   2.31 W
--------------------------------------------
Training Accuracy ($R^2$): 99.98%
Testing Accuracy ($R^2$):  99.94%
